# Google Gemini (google-genai) 사용 예시

> 우리 프로젝트의 멀티AI에서 Gemini를 활용하는 방법
> 
> - 라이브러리: `google-generativeai`
> - 모델: `gemini-2.5-flash` (현재 사용 중)

In [ ]:
# pip install google-generativeai
import google.generativeai as genai
import os

# API 키 설정 (.env에서 가져오거나 직접 입력)
# genai.configure(api_key="YOUR_API_KEY")
genai.configure(api_key=os.getenv('GEMINI_API_KEY', 'YOUR_KEY_HERE'))

print('Gemini API 설정 완료')

## 1. 기본 텍스트 생성

In [ ]:
model = genai.GenerativeModel('gemini-2.5-flash-preview-04-17')

# 간단한 질문
response = model.generate_content('AAPL 주식의 현재 투자 전망을 한국어로 3줄로 요약해줘')
print(response.text)

## 2. 펀더멘탈 데이터 기반 분석 (우리 프로젝트 방식)

In [ ]:
import yfinance as yf

# yfinance에서 실데이터 가져오기
stock = yf.Ticker('AAPL')
info = stock.info

# 펀더멘탈 컨텍스트 구성
fundamental_context = f"""
종목: {info.get('longName')} ({info.get('symbol')})
섹터: {info.get('sector')} / {info.get('industry')}

가치 평가:
- PER: {info.get('trailingPE', 'N/A')}
- Forward PER: {info.get('forwardPE', 'N/A')}
- PBR: {info.get('priceToBook', 'N/A')}
- EV/EBITDA: {info.get('enterpriseToEbitda', 'N/A')}

수익성:
- ROE: {info.get('returnOnEquity', 'N/A')}
- ROA: {info.get('returnOnAssets', 'N/A')}
- 영업이익률: {info.get('operatingMargins', 'N/A')}
- 순이익률: {info.get('profitMargins', 'N/A')}

성장성:
- 매출 성장률: {info.get('revenueGrowth', 'N/A')}
- 이익 성장률: {info.get('earningsGrowth', 'N/A')}

재무 건전성:
- 부채비율: {info.get('debtToEquity', 'N/A')}
- 유동비율: {info.get('currentRatio', 'N/A')}
- 잉여현금흐름: ${info.get('freeCashflow', 0):,.0f}

시장:
- 시가총액: ${info.get('marketCap', 0):,.0f}
- 52주 범위: ${info.get('fiftyTwoWeekLow', 0):.2f} ~ ${info.get('fiftyTwoWeekHigh', 0):.2f}
- 배당수익률: {info.get('dividendYield', 'N/A')}
- 애널리스트 목표가: ${info.get('targetMeanPrice', 'N/A')}
"""

print(fundamental_context)

In [ ]:
# Gemini에 펀더멘탈 데이터를 넣어서 분석 요청
prompt = f"""
당신은 전문 주식 애널리스트입니다. 아래 펀더멘탈 데이터를 기반으로 이 종목의 투자 매력도를 분석해주세요.

분석 항목:
1. 밸류에이션 (PER/PBR 기준 고평가/저평가 판단)
2. 수익성 (ROE/영업이익률 기반 경쟁력)
3. 성장성 (매출/이익 성장 추세)
4. 재무 안정성 (부채비율/현금흐름)
5. 종합 의견 (매수/보유/매도 중 하나 + 근거)

한국어로 간결하게 분석해주세요.

{fundamental_context}
"""

response = model.generate_content(prompt)
print(response.text)

## 3. 채팅 모드 (Multi-turn Conversation)

In [ ]:
# 채팅 세션 - 후속 질문 가능
chat = model.start_chat(history=[])

# 첫 번째 질문
r1 = chat.send_message(f'다음 종목의 펀더멘탈을 분석해줘:\n{fundamental_context}')
print('=== 첫 번째 응답 ===')
print(r1.text[:500])

# 후속 질문 (이전 맥락 유지)
r2 = chat.send_message('이 종목의 가장 큰 리스크 요인 3가지는?')
print('\n=== 후속 질문 응답 ===')
print(r2.text[:500])

## 4. 우리 프로젝트에서의 실제 사용 (trading-monitor/app/api/ai/chat/route.ts)

```typescript
// Gemini 호출 부분 (실제 코드)
async function callGemini(prompt: string): Promise<string> {
  const key = process.env.GEMINI_API_KEY;
  const res = await fetch(
    `https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key=${key}`,
    {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify({
        contents: [{ parts: [{ text: prompt }] }],
      }),
    }
  );
  const data = await res.json();
  return data.candidates?.[0]?.content?.parts?.[0]?.text || '';
}
```

### 환경 변수
```
# trading-monitor/.env
GEMINI_API_KEY=your_key_here
```

## 5. 요약

| 기능 | 메서드 | 용도 |
|------|--------|------|
| 텍스트 생성 | `model.generate_content()` | 단일 분석 요청 |
| 채팅 | `model.start_chat()` | 후속 질문, 맥락 유지 |
| 스트리밍 | `model.generate_content(stream=True)` | 실시간 응답 표시 |
| 이미지 분석 | `model.generate_content([image, prompt])` | 차트 이미지 분석 |

### 우리 프로젝트 적용:
- **현재**: REST API로 직접 호출 (TypeScript)
- **개선**: Python SDK로 더 풍부한 기능 활용 가능
- **펀더멘탈 연동**: yfinance 데이터 → 프롬프트에 주입 → AI 분석